# Giai đoạn 1: Khám phá dữ liệu (Exploratory Data Analysis)
**Dự án:** Phân tích Dữ liệu Thương mại Điện tử Olist (Brazil, 2016–2018)

### Mục tiêu cốt lõi:
1. **Nắm quy mô dữ liệu:** Ghi tổng số dòng và số cột của từng bảng trong 9 bảng dữ liệu gốc, làm mốc "trước khi làm sạch".
2. **Xác định kiểu dữ liệu của từng thuộc tính:** Với mỗi cột, ghi kiểu đang lưu và phân loại theo ý nghĩa (Numerical, Categorical, Datetime, Boolean, định danh); chỉ ra các cột lưu sai kiểu.
3. **Mô tả đặc điểm các biến bằng thống kê mô tả:**
   - Biến số: mean, median, standard deviation, max, min (bổ sung variance).
   - Biến phân loại: số giá trị khác nhau, mode, tần suất (bổ sung mode cho biến số rời rạc như `review_score`).
4. **Ghi nhận quan sát định tính ban đầu:** Nêu các đặc điểm và quy luật nhìn thấy trong dữ liệu, mỗi quan sát kèm con số cụ thể.

## Cấu hình Môi trường & Nạp 9 Bảng Dữ Liệu Gốc
- Thiết lập đường dẫn linh hoạt tự động tìm thư mục `data/raw/`.
- Đọc mã bưu chính (`zip_code_prefix`) ở dạng chuỗi ký tự (`dtype=str`) để bảo toàn các số 0 ở đầu.

In [2]:
import os
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd

# 1. Cấu hình hiển thị và định dạng số thập phân
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# 2. Xác định thư mục gốc dự án linh hoạt (chứa thư mục data/raw)
BASE_DIR = Path("C:/ADY291c-AI2107-Group2")
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
print(f"Thư mục dữ liệu gốc (Raw Data): {RAW_DATA_DIR}")

# 3. Bản đồ 9 file dữ liệu thô
raw_files_map = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

# 4. Đọc dữ liệu, giữ nguyên mã zip code ở dạng chuỗi ký tự
zip_dtype = {
    'customer_zip_code_prefix': str,
    'seller_zip_code_prefix': str,
    'geolocation_zip_code_prefix': str
}

dfs: Dict[str, pd.DataFrame] = {}
for name, filename in raw_files_map.items():
    fpath = RAW_DATA_DIR / filename
    if not fpath.is_file():
        raise FileNotFoundError(f"Không tìm thấy file: {filename} tại {RAW_DATA_DIR}")
    dfs[name] = pd.read_csv(fpath, dtype=zip_dtype)

print("Đã nạp thành công 9 bảng dữ liệu Olist vào bộ nhớ.")

Thư mục dữ liệu gốc (Raw Data): C:\ADY291c-AI2107-Group2\data\raw
Đã nạp thành công 9 bảng dữ liệu Olist vào bộ nhớ.


## 1: Quy Mô Dữ Liệu (Số Dòng, Số Cột Từng Bảng)
Sử dụng thuộc tính `df.shape` để tổng hợp số dòng và số cột của cả 9 bảng dữ liệu thô vào một bảng tóm tắt 9 dòng duy nhất.

In [3]:
# Bước 1: Sử dụng df.shape để tóm tắt quy mô 9 bảng dữ liệu
table_shapes = []
for name, df in dfs.items():
    n_rows, n_cols = df.shape
    table_shapes.append({
        'Tên Bảng Dữ Liệu': f"olist_{name}_dataset" if name != 'category_translation' else 'product_category_name_translation',
        'Số Dòng (Rows)': f"{n_rows:,}",
        'Số Cột (Columns)': n_cols
    })

df_shapes_summary = pd.DataFrame(table_shapes)
print("--- BẢNG TÓM TẮT QUY MÔ 9 BẢNG DỮ LIỆU GỐC ---")
display(df_shapes_summary)

--- BẢNG TÓM TẮT QUY MÔ 9 BẢNG DỮ LIỆU GỐC ---


,Tên Bảng Dữ Liệu,Số Dòng (Rows),Số Cột (Columns)
0,olist_orders_dataset,"99,441",8
1,olist_order_items_dataset,"112,650",7
2,olist_payments_dataset,"103,886",5
3,olist_reviews_dataset,"99,224",7
4,olist_products_dataset,"32,951",9
5,olist_customers_dataset,"99,441",5
6,olist_sellers_dataset,"3,095",4
7,olist_geolocation_dataset,"1,000,163",5
8,product_category_name_translation,71,2


## 2: Kiểu Dữ Liệu của Từng Thuộc Tính (Data Profiling & Classification)
Khảo sát toàn bộ các thuộc tính bằng `df.dtypes` / `df.info()`, phân loại từng cột thành:
- **ID Key (Định danh):** Các cột khóa chính, khóa ngoại hoặc mã bưu chính (`order_id`, `customer_id`, `product_id`, `seller_id`, mã zip).
- **Numerical:** Biến số thực / số nguyên đo lường định lượng (`price`, `freight_value`, `payment_value`, kích thước, khối lượng, điểm đánh giá).
- **Categorical:** Biến phân loại / chuỗi văn bản (`order_status`, `payment_type`, bang, thành phố, tên danh mục).
- **Datetime:** Các mốc thời gian diễn ra vòng đời đơn hàng và đánh giá.
- **Boolean:** Các biến cờ nhị phân (nếu có).


In [4]:
# Bước 2: Kiểm tra df.dtypes và phân loại ngữ nghĩa cho toàn bộ 52 thuộc tính
columns_profile = []

for table_name, df in dfs.items():
    for col, dtype in df.dtypes.items():
        dtype_str = str(dtype)
        
        # Phân loại nhóm kiểu ngữ nghĩa (Semantic Type)
        if any(t_key in col.lower() for t_key in ['date', 'timestamp']) or col.lower().endswith('_at'):
            semantic_group = 'Datetime'
        elif col.endswith('_id') or 'zip_code' in col.lower() or col in ['order_item_id', 'payment_sequential']:
            semantic_group = 'ID Key '
        elif pd.api.types.is_numeric_dtype(dtype):
            if col == 'review_score':
                semantic_group = 'Numerical (Ordinal)'
            else:
                semantic_group = 'Numerical'
        elif pd.api.types.is_bool_dtype(dtype):
            semantic_group = 'Boolean'
        else:
            semantic_group = 'Categorical'
            
        columns_profile.append({
            'Tên Bảng': table_name,
            'Tên Cột (Thuộc tính)': col,
            'Kiểu Lưu Gốc (Raw Dtype)': dtype_str,
            'Nhóm Kiểu Ngữ Nghĩa': semantic_group
        })

df_columns_profile = pd.DataFrame(columns_profile)

# 1. Thống kê tóm tắt số lượng cột theo từng nhóm kiểu ngữ nghĩa
print("--- THỐNG KÊ PHÂN BỐ CÁC NHÓM KIỂU DỮ LIỆU ---")
display(df_columns_profile['Nhóm Kiểu Ngữ Nghĩa'].value_counts().to_frame(name='Số Lượng Thuộc Tính'))

# 2. Hiển thị bảng chi tiết Data Dictionary của toàn bộ 52 thuộc tính
print(f"\n--- BẢNG DATA DICTIONARY CHI TIẾT ({len(df_columns_profile)} THUỘC TÍNH) ---")
display(df_columns_profile)

--- THỐNG KÊ PHÂN BỐ CÁC NHÓM KIỂU DỮ LIỆU ---


,Số Lượng Thuộc Tính
Nhóm Kiểu Ngữ Nghĩa,
ID Key,17
Categorical,13
Numerical,13
Datetime,8
Numerical (Ordinal),1



--- BẢNG DATA DICTIONARY CHI TIẾT (52 THUỘC TÍNH) ---


,Tên Bảng,Tên Cột (Thuộc tính),Kiểu Lưu Gốc (Raw Dtype),Nhóm Kiểu Ngữ Nghĩa
0,orders,order_id,str,ID Key
1,orders,customer_id,str,ID Key
2,orders,order_status,str,Categorical
3,orders,order_purchase_timestamp,str,Datetime
4,orders,order_approved_at,str,Datetime
5,orders,order_delivered_carrier_date,str,Datetime
6,orders,order_delivered_customer_date,str,Datetime
7,orders,order_estimated_delivery_date,str,Datetime
8,order_items,order_id,str,ID Key
9,order_items,order_item_id,int64,ID Key


### Ghi Chú Đánh Giá Kỹ Thuật (Technical Notes)
Bảng data dictionary phía trên chỉ giữ 4 cột mô tả cấu trúc dữ liệu; các ghi chú đánh giá kỹ thuật theo từng nhóm kiểu ngữ nghĩa được tổng hợp tại đây:

- **Datetime (8 cột): ĐANG SAI KIỂU** - dữ liệu đang lưu dạng text (`object`), cần ép sang `datetime64[ns]` ở Giai đoạn 2 để tính thời gian giao vận và trễ hạn:
  - `orders.order_purchase_timestamp`, `orders.order_approved_at`, `orders.order_delivered_carrier_date`, `orders.order_delivered_customer_date`, `orders.order_estimated_delivery_date`.
  - `order_items.shipping_limit_date`, `reviews.review_creation_date`, `reviews.review_answer_timestamp`.
- **ID Key (Định danh) - 17 cột:** Khóa chính/khóa ngoại dùng để nối bảng; riêng 3 cột mã bưu chính phải giữ dạng chuỗi `zfill(5)` để bảo toàn các số 0 ở đầu: `customers.customer_zip_code_prefix`, `sellers.seller_zip_code_prefix`, `geolocation.geolocation_zip_code_prefix`.
- **Numerical (Số học) - 13 cột:** Chuẩn định lượng số thực/nguyên, sẵn sàng cho thống kê mô tả và tính toán chỉ số.
- **Numerical (Ordinal) - 1 cột:** `review_score` là biến thứ bậc 1-5 sao và là biến mục tiêu (target) của dự án.
- **Categorical (Phân loại) - 13 cột:** Chuẩn biến định tính / chuỗi văn bản, cần mã hóa (encoding) trước khi đưa vào mô hình.
- **Boolean - 0 cột:** Không có biến cờ nhị phân Đúng/Sai trong 9 bảng dữ liệu gốc.


## 3: Thống Kê Mô Tả (Descriptive Statistics - Chuẩn CLO6)
Trình bày kết quả thống kê mô tả hai tầng bằng hai bảng tổng hợp tinh gọn, không dán nguyên bảng đầu ra thô dài dòng:
1. **Biến định lượng (Numerical Variables):** Sử dụng `df.describe()` kết hợp tính toán thêm **Variance (Phương sai)** và **Mode (Yếu vị)** theo yêu cầu chuẩn CLO6 cho 5 thuộc tính chủ chốt gắn với mục tiêu dự đoán Review Score: `price`, `freight_value`, `payment_value`, `payment_installments`, `review_score`.
2. **Biến định tính (Categorical Variables):** Sử dụng `df.describe(include='object')` và `value_counts()` khảo sát số lượng giá trị phân biệt (`Unique`), giá trị xuất hiện nhiều nhất (`Mode`), tần suất (`Frequency`) và tỷ lệ % của mode cho 3 thuộc tính: `order_status`, `payment_type`, `customer_state`.

In [ ]:
# Bước 3.1: Thống kê mô tả Biến Số Học (bổ sung Variance và Mode theo chuẩn CLO6)
numeric_vars = {
    'price': dfs['order_items']['price'],
    'freight_value': dfs['order_items']['freight_value'],
    'payment_value': dfs['payments']['payment_value'],
    'payment_installments': dfs['payments']['payment_installments'],
    'review_score': dfs['reviews']['review_score']
}

num_summary_records = []
for var_name, series in numeric_vars.items():
    s_valid = series.dropna()
    mode_val = s_valid.mode()[0] if not s_valid.empty else np.nan
    
    num_summary_records.append({
        'Tên Biến': var_name,
        'Count': f"{int(s_valid.count()):,}",
        'Mean': f"{s_valid.mean():,.2f}",
        'Median': f"{s_valid.median():,.2f}",
        'Std': f"{s_valid.std():,.2f}",
        'Variance': f"{s_valid.var():,.2f}",
        'Mode': f"{mode_val:,.2f}",
        'Min': f"{s_valid.min():,.2f}",
        'Max': f"{s_valid.max():,.2f}"
    })

df_num_summary = pd.DataFrame(num_summary_records)
print("--- BẢNG THỐNG KÊ MÔ TẢ BIẾN ĐỊNH LƯỢNG (NUMERICAL) ---")
display(df_num_summary)

--- BẢNG THỐNG KÊ MÔ TẢ BIẾN ĐỊNH LƯỢNG (NUMERICAL - CLO6) ---


,Tên Biến,Count,Mean,Median,Std,Variance,Mode,Min,Max
0,price,"112,650",120.65,74.99,183.63,"33,721.42",59.90,0.85,"6,735.00"
1,freight_value,"112,650",19.99,16.26,15.81,249.84,15.10,0.00,409.68
2,payment_value,"103,886",154.10,100.00,217.49,"47,303.67",50.00,0.00,"13,664.08"
3,payment_installments,"103,886",2.85,1.00,2.69,7.22,1.00,0.00,24.00
4,review_score,"99,224",4.09,5.00,1.35,1.82,5.00,1.00,5.00


### Nhận xét thống kê mô tả

- **`price`:** Mean (120.65) cao hơn Median (74.99), độ lệch chuẩn lớn (183.63) và Max đạt 6,735.00. Dữ liệu có dấu hiệu phân phối lệch phải và khả năng xuất hiện giá trị ngoại lai.

- **`freight_value`:** Mean (19.99) cao hơn Median (16.26), Max đạt 409.68. Một số đơn hàng có phí vận chuyển cao bất thường cần được kiểm tra thêm.

- **`payment_value`:** Mean (154.10) cao hơn Median (100.00), độ lệch chuẩn lớn (217.49) và Max đạt 13,664.08. Giá trị thanh toán có độ phân tán lớn, cần kiểm tra các giá trị cực đoan.

- **`payment_installments`:** Median và Mode đều bằng 1, nhưng Max đạt 24 và Min bằng 0. Cần kiểm tra tính hợp lệ của số kỳ thanh toán.

- **`review_score`:** Mean đạt 4.09, Median và Mode đều bằng 5. Điều này cho thấy phần lớn khách hàng có xu hướng đánh giá tích cực.

**Định hướng tiền xử lý:** Các biến `price`, `freight_value` và `payment_value` cần được kiểm tra ngoại lai bằng Boxplot và phương pháp IQR. Riêng `payment_installments` cần kiểm tra giá trị không hợp lệ. Không mặc định mọi giá trị cực đoan đều là dữ liệu sai.

In [6]:
# Bước 3.2: Thống kê mô tả Biến Phân Loại (Categorical Variables)
key_cat_targets = [
    ('orders', 'order_status', 'Trạng thái đơn hàng'),
    ('payments', 'payment_type', 'Phương thức thanh toán'),
    ('customers', 'customer_state', 'Bang của khách hàng')
]

cat_summary_records = []
for tbl_name, col_name, desc in key_cat_targets:
    s_valid = dfs[tbl_name][col_name].dropna()
    val_counts = s_valid.value_counts()
    top_mode = val_counts.index[0] if len(val_counts) > 0 else 'N/A'
    mode_freq = val_counts.iloc[0] if len(val_counts) > 0 else 0
    mode_pct = (mode_freq / len(s_valid) * 100) if len(s_valid) > 0 else 0.0
    
    cat_summary_records.append({
        'Bảng Dữ Liệu': tbl_name,
        'Tên Biến': col_name,
        'Mô Tả Thuộc Tính': desc,
        'Số Giá Trị Khác Nhau (Unique)': f"{s_valid.nunique():,}",
        'Mode (Phổ biến nhất)': top_mode,
        'Tần Suất Mode (Frequency)': f"{mode_freq:,}",
        'Tỷ Lệ Mode (%)': f"{mode_pct:.2f}%"
    })

df_cat_summary = pd.DataFrame(cat_summary_records)
print("--- BẢNG THỐNG KÊ MÔ TẢ BIẾN ĐỊNH TÍNH (CATEGORICAL) ---")
display(df_cat_summary)

--- BẢNG THỐNG KÊ MÔ TẢ BIẾN ĐỊNH TÍNH (CATEGORICAL) ---


,Bảng Dữ Liệu,Tên Biến,Mô Tả Thuộc Tính,Số Giá Trị Khác Nhau (Unique),Mode (Phổ biến nhất),Tần Suất Mode (Frequency),Tỷ Lệ Mode (%)
0,orders,order_status,Trạng thái đơn hàng,8,delivered,"96,478",97.02%
1,payments,payment_type,Phương thức thanh toán,5,credit_card,"76,795",73.92%
2,customers,customer_state,Bang của khách hàng,27,SP,"41,746",41.98%



- **`order_status`:** Trạng thái `delivered` chiếm ưu thế với 97.02%, cho thấy phần lớn đơn hàng trong dữ liệu đã được giao thành công.
- **`payment_type`:** `credit_card` là phương thức thanh toán phổ biến nhất, chiếm 73.92%, cho thấy khách hàng chủ yếu thanh toán bằng thẻ tín dụng.
- **`customer_state`:** Có 27 bang khác nhau, trong đó SP chiếm tỷ lệ cao nhất (41.98%), cho thấy dữ liệu khách hàng tập trung đáng kể tại bang São Paulo.
